In [1]:
import pandas as pd

# Load the JSON file
df = pd.read_json('ethno_sample_400.json')

print("Total rows in original file:", len(df))

# Drop rows where the SMILES string is missing
df_clean = df.dropna(subset=['canonical_smiles']).copy()
print("Total rows after removing missing SMILES:", len(df_clean))

Total rows in original file: 400
Total rows after removing missing SMILES: 293


In [2]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

def smiles_to_fingerprint(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is not None:
        # Generate Morgan Fingerprint (radius 2, 1024 bits)
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
        return np.array(fp)
    return None

print("Generating fingerprints (this might take 10-20 seconds)...")

# Apply the function to our clean dataframe
df_clean['fingerprint'] = df_clean['canonical_smiles'].apply(smiles_to_fingerprint)

# Drop any rows where RDKit failed to read the SMILES
df_clean = df_clean.dropna(subset=['fingerprint'])

print(f"Successfully generated fingerprints for {len(df_clean)} compounds!")

Generating fingerprints (this might take 10-20 seconds)...


[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerat

Successfully generated fingerprints for 293 compounds!


[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator
[22:07:51] DEPRECATION WARNING: please use MorganGenerator


In [3]:
!pip install rdkit

In [4]:
from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

def smiles_to_fingerprint(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is not None:
        # Generate Morgan Fingerprint (radius 2, 1024 bits)
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024)
        return np.array(fp)
    return None

print("Generating fingerprints (this might take 10-20 seconds)...")

# Apply the function to our clean dataframe
df_clean['fingerprint'] = df_clean['canonical_smiles'].apply(smiles_to_fingerprint)

# Drop any rows where RDKit failed to read the SMILES
df_clean = df_clean.dropna(subset=['fingerprint'])

print(f"Successfully generated fingerprints for {len(df_clean)} compounds!")

Generating fingerprints (this might take 10-20 seconds)...


[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerator
[22:07:54] DEPRECATION WARNING: please use MorganGenerat

Successfully generated fingerprints for 293 compounds!


In [5]:
from sklearn.cluster import KMeans
import numpy as np

# 1. Convert the fingerprints into a matrix
X = np.array(list(df_clean['fingerprint']))

# 2. Train the ML model to find 3 chemical groups (clusters)
# random_state=42 just ensures the result is the same every time you run it
kmeans = KMeans(n_clusters=3, random_state=42)
kmeans.fit(X)

# 3. Assign each compound to its cluster
df_clean['cluster'] = kmeans.labels_

print("Machine Learning training complete!")
print("Number of compounds in each cluster:")
print(df_clean['cluster'].value_counts())

Machine Learning training complete!
Number of compounds in each cluster:
cluster
2    199
1     91
0      3
Name: count, dtype: int64


D:\Users\TECHTREASURE\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1428: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=2.
  warnings.warn(


In [6]:
# Let's look at the top 3 compounds from each cluster
for i in range(3):
    print(f"\n--- Cluster {i} Compounds ---")
    cluster_compounds = df_clean[df_clean['cluster'] == i]['chemical'].head(3).tolist()
    for comp in cluster_compounds:
        print("-", comp)


--- Cluster 0 Compounds ---
- EUGLOBAL-VII
- CARNOSIC-ACID
- CARNOSOLIC-ACID

--- Cluster 1 Compounds ---
- MASLINIC-ACID
- PIPERITENONE
- TERPINEN-4-OL

--- Cluster 2 Compounds ---
- UNDECAN-2-ONE
- OCTANOIC-ACID
- SALIDROSIDE


In [7]:
# Known Mpro Inhibitors (Actives)
active_smiles = [
    "CC1(C2C1C3C(C3(C)C)CC2=O)C", # Nirmatrelvir fragment (simplified)
    "C1=CC=C(C=C1)C2=CC(=O)C3=C(C=C(C=C3O2)O)O", # Quercetin
    "CC1=CC(=O)C(=CC1=O)O", # Simple Quinone
    "C1CC1C(=O)NC2=CC=CC=C2", # Simple amide
    "CC(C)C1=CC=CC=C1" # Simple aromatic
]

# Known Inactives (Random simple molecules like sugars/alcohols)
inactive_smiles = [
    "CCO", # Ethanol
    "CC(=O)O", # Acetic acid
    "C(C(C(=O)O)O)O", # Simple sugar
    "C1CCCCC1", # Cyclohexane
    "CCCCCC" # Hexane
]

# Combine them into a training dataset
train_smiles = active_smiles + inactive_smiles
train_labels = [1]*len(active_smiles) + [0]*len(inactive_smiles) # 1 = Active, 0 = Inactive

print("Mini training set created with", len(train_smiles), "compounds.")

Mini training set created with 10 compounds.


In [8]:
from sklearn.ensemble import RandomForestClassifier

# Generate fingerprints for the training set
def get_fp(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return np.array(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=1024))

X_train = np.array([get_fp(s) for s in train_smiles])
y_train = np.array(train_labels)

# Train the Random Forest model
print("Training the Random Forest Classifier...")
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
print("Model trained successfully!")

Training the Random Forest Classifier...
Model trained successfully!


[22:08:02] DEPRECATION WARNING: please use MorganGenerator
[22:08:02] DEPRECATION WARNING: please use MorganGenerator
[22:08:02] DEPRECATION WARNING: please use MorganGenerator
[22:08:02] DEPRECATION WARNING: please use MorganGenerator
[22:08:02] DEPRECATION WARNING: please use MorganGenerator
[22:08:02] DEPRECATION WARNING: please use MorganGenerator
[22:08:02] DEPRECATION WARNING: please use MorganGenerator
[22:08:02] DEPRECATION WARNING: please use MorganGenerator
[22:08:02] DEPRECATION WARNING: please use MorganGenerator
[22:08:02] DEPRECATION WARNING: please use MorganGenerator


In [9]:
# Use our trained model to predict the 293 plant compounds
X_plants = np.array(list(df_clean['fingerprint']))

# Predict the probability of being an Mpro inhibitor (1 = high probability)
probabilities = rf_model.predict_proba(X_plants)[:, 1]

# Add the predictions to our dataframe
df_clean['mpro_probability'] = probabilities

# Sort the compounds to find the top 10 best candidates
top_10_plants = df_clean.sort_values(by='mpro_probability', ascending=False).head(10)

print("=== TOP 10 PLANT COMPOUNDS PREDICTED TO INHIBIT Mpro ===")
for index, row in top_10_plants.iterrows():
    print(f"{row['chemical']}: {row['mpro_probability']*100:.2f}% probability")

=== TOP 10 PLANT COMPOUNDS PREDICTED TO INHIBIT Mpro ===
APIGENIN: 70.00% probability
BAICALIN: 69.00% probability
BETAVULGARIN: 69.00% probability
DIHYDROQUERCETIN: 63.00% probability
DIHYDROQUERCETIN: 63.00% probability
CHRYSOPHANOL: 60.00% probability
COUMARIN: 59.00% probability
METHYLEPHEDRINE: 59.00% probability
QUERCETIN: 59.00% probability
QUERCETIN: 59.00% probability


In [10]:
# Drop duplicates based on the chemical name so each compound only appears once
df_unique = df_clean.drop_duplicates(subset=['chemical'])

# Now sort and find the top 10 from the unique list
top_10_unique = df_unique.sort_values(by='mpro_probability', ascending=False).head(10)

print("=== TOP 10 UNIQUE COMPOUNDS ===")
for index, row in top_10_unique.iterrows():
    print(f"{row['chemical']}: {row['mpro_probability']*100:.2f}%")

=== TOP 10 UNIQUE COMPOUNDS ===
APIGENIN: 70.00%
BAICALIN: 69.00%
BETAVULGARIN: 69.00%
DIHYDROQUERCETIN: 63.00%
CHRYSOPHANOL: 60.00%
COUMARIN: 59.00%
METHYLEPHEDRINE: 59.00%
QUERCETIN: 59.00%
RHODOXANTHIN: 58.00%
5-0-METHYLGENISTEIN: 58.00%
